# 03 — NLP Pathology Label Imputation

GTEx provides a structured `Pathology.Categories` field for only ~32% of samples; the remaining 68% have findings buried in the free-text `Pathology.Notes`. This notebook recovers structured labels from those notes using regex pattern matching, ConText-style negation, and percentage-based severity grading.

**Why two output matrices.** Binary indicates *whether* a finding is present per subject and tissue group; continuous gives the *grade* (0–100 percent), needed for downstream regression on severity rather than presence alone.

**Source of truth.** All patterns, helpers, and severity logic live in [`gtex_biomarkers/labels_v2.py`](../gtex_biomarkers/labels_v2.py) — review or edit there. This notebook imports from that module and runs the pipeline end-to-end.

**Vocabulary.** 79 patterns total: 57 covering GTEx's controlled `Pathology.Categories` vocabulary, plus 22 additional concepts that appear in free-text notes but were never structure-coded (`plaque`, `intimal_thickening`, `fatty_infiltration`, `medial_degeneration`, `bronchitis`, `colitis`, `amyloidosis`, `regressive_change`, etc.).

**Outputs** (saved to `data/processed/`):
- `liver_pathology_labels_imputed.csv` — liver-only legacy long-form table with the comma-joined `Pathology.Categories.Final` column (consumed by NB04).
- `gtex_pathology_matrix_binary.csv` — wide binary matrix, `subject_id` × `tissue_group.term`, values in {0, 1, NaN}.
- `gtex_pathology_matrix_continuous.csv` — wide percentage-severity matrix (0–100), same shape, NaN where no severity signal was extracted.

In [1]:
from collections import defaultdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from gtex_biomarkers.config import Config
from gtex_biomarkers.labels_v2 import (
    _RAW_PATTERNS,
    extract_categories_v2 as extract_categories,
    extract_severities,
)

print(f'Loaded {len(_RAW_PATTERNS)} category patterns from labels_v2.py')

Loaded 79 category patterns from labels_v2.py


## 1. Inspect the pattern vocabulary

The 79 category patterns are defined in `labels_v2.py`. Each entry has `pat` (case-insensitive regex), `exclude` (peer categories that suppress this one if they co-occur in the same subclause), and `skip_negation` (True for self-negating labels like `no_abnormalities` where the negation IS the label). Below: all category names plus three example rule definitions for reference. Edit the rules in `labels_v2.py` — this cell is read-only inspection.

In [2]:
print('Categories (alphabetical):')
for cat in sorted(_RAW_PATTERNS):
    print(f'  {cat}')

print('\nSample patterns (3 examples):')
for cat in ['steatosis', 'sertoli_only', 'plaque']:
    print(f'\n  {cat}:')
    print(f'    pat     = {_RAW_PATTERNS[cat]["pat"]!r}')
    print(f'    exclude = {_RAW_PATTERNS[cat].get("exclude", [])}')
    print(f'    skip_negation = {_RAW_PATTERNS[cat].get("skip_negation", False)}')

Categories (alphabetical):
  adenoma
  amylacea
  amyloidosis
  atelectasis
  atherosclerosis
  atherosis
  atrophy
  bronchitis
  calcification
  cholesterol_clefts
  cirrhosis
  clean_specimens
  colitis
  congestion
  consolidation
  corpora_albicantia
  cyst
  desquamation
  diabetic
  dysplasia
  edema
  emphysema
  endometrium_change
  esophagitis
  fatty_infiltration
  fiber_degeneration
  fibrosis
  foreign_body
  gastritis
  glomerulosclerosis
  goiter
  gynecomastoid
  hashimoto
  heart_failure_cells
  hemorrhage
  hepatitis
  hepatocyte_degeneration
  hyalinization
  hypereosinophilia
  hyperplasia
  hypertrophy
  hypoxic
  infarction
  inflammation
  intimal_thickening
  ischemic_changes
  leiomyoma
  macrophages
  mastopathy
  medial_degeneration
  metaplasia
  monckeberg
  necrosis
  neoplasm
  nephritis
  nephrosclerosis
  neuroendocrine_tumor
  no_abnormalities
  nodularity
  pancreatitis
  pigment
  plaque
  pneumonia
  post_menopausal
  prostatitis
  regressive_change

## 2. Smoke-test on a few real notes

Six diagnostic cases that exercise the three trickiest behaviours: explicit percentages (`60% steatosis` → severity 60), negation (`no evidence of hypoxic damage` → no label), and descriptive variants (`only Sertoli cells remain` → `sertoli_only`; `Hashimoto thyroiditis` → `hashimoto` + `inflammation`). If any of these print unexpected output after a pattern edit, fix the regex in `labels_v2.py` before re-running the rest of the notebook.

In [3]:
for note in [
    "2 pieces, 60% steatosis, mild fibrosis",
    "6 pieces; severe (diabetic) glomerulosclerosis",
    "atheromatous plaque ~25% luminal compromise, cholesterol clefts",
    "purkinje cells, no evidence of hypoxic damage",
    "2 pieces; only Sertoli cells remain (germ cell aplasia)",
    "2 pieces; Hashimoto thyroiditis with moderate fibrosis",
]:
    print(note)
    print("  cats:", extract_categories(note))
    print("  sev :", extract_severities(note))

2 pieces, 60% steatosis, mild fibrosis
  cats: [('fibrosis', 0.9), ('steatosis', 0.9)]
  sev : {'steatosis': 60.0, 'fibrosis': 15.0}
6 pieces; severe (diabetic) glomerulosclerosis
  cats: [('diabetic', 0.9), ('glomerulosclerosis', 0.9), ('sclerotic', 0.7)]
  sev : {'glomerulosclerosis': 70.0, 'diabetic': 70.0}
atheromatous plaque ~25% luminal compromise, cholesterol clefts
  cats: [('plaque', 0.9), ('cholesterol_clefts', 0.7)]
  sev : {'plaque': 25.0}
purkinje cells, no evidence of hypoxic damage
  cats: []
  sev : {}
2 pieces; only Sertoli cells remain (germ cell aplasia)
  cats: [('sertoli_only', 0.7)]
  sev : {}
2 pieces; Hashimoto thyroiditis with moderate fibrosis
  cats: [('fibrosis', 0.9), ('hashimoto', 0.9), ('inflammation', 0.9)]
  sev : {'fibrosis': 35.0, 'inflammation': 35.0, 'hashimoto': 35.0}


## 3. Validate on GTEx ground-truth `Pathology.Categories`

Of 25,713 samples, ~8,100 have a non-null structured category field that GTEx pathologists manually filled in alongside the notes. We treat that structured field as ground truth: for those samples, run our NLP on the **notes** and score per-term recall / precision / F1 against the **structured labels**.

**Reading the output.** The validation vocab printed below is **57 tokens**, not 79 — that is the size of the GTEx structured vocabulary, *not our pattern dictionary*. Our 79 patterns include 22 concepts (`plaque`, `intimal_thickening`, `fatty_infiltration`, `medial_degeneration`, etc.) that appear in notes but were never structure-coded, so there is no ground-truth column to validate them against. Those patterns still run and populate the output matrices — they just don't appear in this validation table.

**Expected micro F1 ≈ 0.94** (recall ≈ 0.95, precision ≈ 0.94).

In [4]:
df_meta_url = pd.read_csv(Config.RAW_DIR / "meta_data_with_url.csv")
df_meta_url["pred_cats"] = df_meta_url["Pathology.Notes"].apply(
    lambda n: set(c for c, _ in extract_categories(n))
)
df_meta_url["pred_sev"] = df_meta_url["Pathology.Notes"].apply(extract_severities)

gt = df_meta_url[df_meta_url["Pathology.Categories"].notna()].copy()
gt["gt_set"] = gt["Pathology.Categories"].apply(
    lambda s: {x.strip().lower() for x in str(s).split(",") if x.strip()}
)
gtex_vocab = sorted({c for s in gt["gt_set"] for c in s})
print(f"GTEx ground-truth samples (with structured Pathology.Categories): {len(gt)}")
print(f"GTEx structured vocabulary size (validation reference): {len(gtex_vocab)} "
      f"-- smaller than our {len(_RAW_PATTERNS)}-pattern dictionary by design (see markdown above).")

rows = []
for term in gtex_vocab:
    in_gt = gt["gt_set"].apply(lambda s: term in s)
    in_pr = gt["pred_cats"].apply(lambda s: term in s)
    TP = int((in_gt & in_pr).sum()); FP = int((~in_gt & in_pr).sum())
    FN = int((in_gt & ~in_pr).sum()); TN = int((~in_gt & ~in_pr).sum())
    rec = TP / (TP + FN) if (TP + FN) else np.nan
    pre = TP / (TP + FP) if (TP + FP) else np.nan
    f1  = 2 * pre * rec / (pre + rec) if (pre and rec and (pre + rec) > 0) else np.nan
    rows.append({"term": term, "support": TP + FN, "TP": TP, "FP": FP, "FN": FN,
                 "recall": round(rec, 3), "precision": round(pre, 3), "f1": round(f1, 3)})
val_df = pd.DataFrame(rows).sort_values("support", ascending=False)
TP_tot = val_df["TP"].sum(); FP_tot = val_df["FP"].sum(); FN_tot = val_df["FN"].sum()
print(f"\nMicro-averaged  recall={TP_tot/(TP_tot+FN_tot):.3f}  precision={TP_tot/(TP_tot+FP_tot):.3f}  F1={2*TP_tot/(2*TP_tot+FP_tot+FN_tot):.3f}")
val_df.head(25)

GTEx ground-truth samples (with structured Pathology.Categories): 8122
GTEx structured vocabulary size (validation reference): 57 -- smaller than our 79-pattern dictionary by design (see markdown above).

Micro-averaged  recall=0.946  precision=0.941  F1=0.944


,term,support,TP,FP,FN,recall,precision,f1
18,fibrosis,1524,1522,125,2,0.999,0.924,0.960
9,congestion,1412,1406,3,6,0.996,0.998,0.997
8,clean_specimens,738,665,12,73,0.901,0.982,0.940
42,no_abnormalities,617,596,5,21,0.966,0.992,0.979
51,sclerotic,489,428,92,61,0.875,0.823,0.848
53,spermatogenesis,487,477,4,10,0.979,0.992,0.986
6,calcification,476,458,34,18,0.962,0.931,0.946
4,atherosis,414,404,18,10,0.976,0.957,0.967
5,atrophy,395,389,43,6,0.985,0.900,0.941
29,hyperplasia,379,359,5,20,0.947,0.986,0.966


## 4. Apply to all 25,713 samples; aggregate per (subject, tissue group)

Sample-level labels are too granular for downstream modelling — most analyses operate per donor. We collapse to **subject × tissue group** (13 organ-system groups defined in `TISSUE_GROUPS` below). Aggregation rules:
- **Binary**: a finding is positive for the (subject, group) cell if it was flagged in *any* sample from that group. NaN if the subject was not sampled in that group at all.
- **Severity**: the max percent across the subject's samples in that group. Pathologists grade the worst-affected region, so the max matches their reporting convention.

In [5]:
TISSUE_GROUPS = {
    "cardiovascular_artery": ["Artery - Aorta", "Artery - Coronary", "Artery - Tibial"],
    "cardiovascular_heart":  ["Heart - Atrial Appendage", "Heart - Left Ventricle"],
    "cns_pns":               ["Brain - Cortex", "Brain - Cerebellum", "Nerve - Tibial", "Pituitary"],
    "endocrine":             ["Adrenal Gland", "Thyroid", "Pituitary"],
    "gi_lower":              ["Colon - Sigmoid", "Colon - Transverse", "Small Intestine - Terminal Ileum"],
    "gi_upper":              ["Esophagus - Mucosa", "Esophagus - Muscularis", "Esophagus - Gastroesophageal Junction", "Stomach"],
    "hepatic":               ["Liver"],
    "pancreatic":            ["Pancreas"],
    "pulmonary":             ["Lung"],
    "renal":                 ["Kidney - Cortex", "Kidney - Medulla", "Bladder"],
    "repro_female":          ["Ovary", "Uterus", "Vagina", "Cervix - Endocervix", "Cervix - Ectocervix",
                              "Fallopian Tube", "Breast - Mammary Tissue"],
    "repro_male":            ["Prostate", "Testis"],
    "soft_tissue":           ["Adipose - Subcutaneous", "Adipose - Visceral (Omentum)", "Muscle - Skeletal",
                              "Skin - Sun Exposed (Lower leg)", "Skin - Not Sun Exposed (Suprapubic)",
                              "Spleen", "Minor Salivary Gland"],
}
tissue_to_group = {t: g for g, ts in TISSUE_GROUPS.items() for t in ts}
df_meta_url["tissue_group"] = df_meta_url["Tissue"].map(tissue_to_group)
print(f"Samples mapped to a tissue group: {df_meta_url['tissue_group'].notna().sum()} / {len(df_meta_url)}")

subj_group_cats: Dict[Tuple[str, str], set] = defaultdict(set)
subj_group_sev:  Dict[Tuple[str, str], Dict[str, float]] = defaultdict(dict)
subj_group_seen: Dict[Tuple[str, str], int] = defaultdict(int)
for subj, grp, cats, sev in zip(df_meta_url["Subject.ID"], df_meta_url["tissue_group"],
                                  df_meta_url["pred_cats"], df_meta_url["pred_sev"]):
    if not isinstance(grp, str):
        continue
    key = (subj, grp)
    subj_group_cats[key] |= cats
    for cat, s in sev.items():
        if s > subj_group_sev[key].get(cat, -1.0):
            subj_group_sev[key][cat] = s
    subj_group_seen[key] += 1
print(f"Subject×group cells with at least one sample: {len(subj_group_seen)}")

Samples mapped to a tissue group: 25713 / 25713
Subject×group cells with at least one sample: 10948


## 5. Build wide binary + continuous matrices

Reshape the aggregated dictionaries into rectangular tables: one row per subject, one column per `{tissue_group}.{term}` pair (only terms that fired at least once in that group get a column — avoids 79×13 = 1,027 mostly-empty columns).

**Cell semantics — both matrices share the same NaN convention:**
- `NaN` (binary or continuous): the subject was **not sampled** in that tissue group, so we have nothing to say about it. Crucial: a NaN is not a negative.
- `0` (binary only): the subject *was* sampled in that group, but our NLP did not flag this term.
- `1` (binary): flagged in at least one sample for that subject × group.
- A `NaN` in **continuous** with a `1` in **binary** means: term present, but no percentage or ordinal qualifier was extractable from the notes (positive without a known grade).

In [6]:
all_subjects = sorted(df_meta_url["Subject.ID"].dropna().unique())

group_terms: Dict[str, set] = defaultdict(set)
for (subj, grp), cats in subj_group_cats.items():
    group_terms[grp] |= cats
group_terms = {g: sorted(ts) for g, ts in group_terms.items()}
binary_cols = ["subject_id"] + [f"{g}.{t}" for g in sorted(group_terms) for t in group_terms[g]]
print(f"Matrix shape: ({len(all_subjects)}, {len(binary_cols)})")

bin_records, cont_records = [], []
for subj in all_subjects:
    brow = {"subject_id": subj}
    crow = {"subject_id": subj}
    for g, terms in group_terms.items():
        sampled = (subj, g) in subj_group_seen
        cats = subj_group_cats.get((subj, g), set())
        sev_map = subj_group_sev.get((subj, g), {})
        for t in terms:
            brow[f"{g}.{t}"] = (1.0 if t in cats else 0.0) if sampled else np.nan
            crow[f"{g}.{t}"] = sev_map.get(t, np.nan)
    bin_records.append(brow)
    cont_records.append(crow)
binary_df = pd.DataFrame.from_records(bin_records, columns=binary_cols)
continuous_df = pd.DataFrame.from_records(cont_records, columns=binary_cols)
print(f"Continuous: {(continuous_df.iloc[:, 1:].notna().sum() > 0).sum()} term-columns with at least one severity value")

Matrix shape: (980, 342)


Continuous: 239 term-columns with at least one severity value


## 6. Imputation coverage gain per tissue × category

How much label coverage did the NLP actually add? For each (Tissue, category), count:
- **`original`** — samples already labelled with this category in GTEx's structured `Pathology.Categories` field.
- **`imputed`** — samples where the structured field was NaN, but our NLP recovered this category from `Pathology.Notes`.
- **`total`** — union (original ∪ imputed).
- **`gain_pct`** — `imputed / total` (how much of the final positive count came from NLP, not from GTEx).

Two tables below: (1) per-tissue rollup showing overall coverage lift, (2) top per-(tissue, category) cells ranked by absolute number of newly imputed positives — the same view we used to confirm the liver `cirrhosis` imputation gain.

In [7]:
# Per-sample original (structured) labels and our NLP predictions
df_meta_url["gt_set"] = df_meta_url["Pathology.Categories"].apply(
    lambda s: {x.strip().lower() for x in str(s).split(",") if x.strip()} if pd.notna(s) else set()
)
has_structured = df_meta_url["Pathology.Categories"].notna()

# ── Table 1: per-tissue rollup ───────────────────────────────────────────
rollup_rows = []
for tissue, sub in df_meta_url.groupby("Tissue"):
    n = len(sub)
    n_struct = int(has_structured.loc[sub.index].sum())
    # Sample-level: any predicted cat at all from NLP
    nlp_any = sub["pred_cats"].apply(len) > 0
    # Samples newly covered (structured was NaN, NLP added at least one cat)
    n_newly_labelled = int(((~has_structured.loc[sub.index]) & nlp_any).sum())
    rollup_rows.append({
        "Tissue": tissue,
        "n_samples": n,
        "structured_only": n_struct,
        "newly_labelled_by_nlp": n_newly_labelled,
        "final_coverage": n_struct + n_newly_labelled,
        "coverage_before_pct": round(100 * n_struct / n, 1),
        "coverage_after_pct": round(100 * (n_struct + n_newly_labelled) / n, 1),
    })
rollup_df = pd.DataFrame(rollup_rows).sort_values("newly_labelled_by_nlp", ascending=False)
print("=== Per-tissue coverage rollup (sorted by newly-labelled samples) ===")
print(rollup_df.to_string(index=False))

=== Per-tissue coverage rollup (sorted by newly-labelled samples) ===
                               Tissue  n_samples  structured_only  newly_labelled_by_nlp  final_coverage  coverage_before_pct  coverage_after_pct
                      Artery - Tibial        979              514                    168             682                 52.5                69.7
                       Artery - Aorta        858              407                    158             565                 47.4                65.9
     Small Intestine - Terminal Ileum        798               39                    139             178                  4.9                22.3
               Adipose - Subcutaneous        978              170                    136             306                 17.4                31.3
              Breast - Mammary Tissue        894              362                    135             497                 40.5                55.6
                   Colon - Transverse        937      

In [8]:
# ── Table 2: per (tissue, category) original vs imputed counts ──────────
detail_rows = []
for tissue, sub in df_meta_url.groupby("Tissue"):
    cats_seen = set()
    for s in sub["gt_set"]:
        cats_seen |= s
    for s in sub["pred_cats"]:
        cats_seen |= s
    for cat in cats_seen:
        original = int(sub["gt_set"].apply(lambda s: cat in s).sum())
        # Imputed = structured NaN AND NLP predicted this cat
        imputed = int(
            (~has_structured.loc[sub.index]
             & sub["pred_cats"].apply(lambda s: cat in s)).sum()
        )
        total = int((sub["gt_set"].apply(lambda s: cat in s)
                     | sub["pred_cats"].apply(lambda s: cat in s)).sum())
        if total == 0:
            continue
        detail_rows.append({
            "Tissue": tissue, "category": cat,
            "original": original, "imputed": imputed, "total": total,
            "gain_pct": round(100 * imputed / total, 1) if total else 0.0,
        })
detail_df = pd.DataFrame(detail_rows)

# Top 30 cells by absolute number of newly imputed positives
print("=== Top 30 (Tissue, category) cells by NLP-imputed positives ===")
print(detail_df.sort_values("imputed", ascending=False)
      .head(30).to_string(index=False))

# Specifically: liver cirrhosis (the reference case)
print("\n=== Reference: Liver × cirrhosis ===")
lc = detail_df[(detail_df["Tissue"] == "Liver") & (detail_df["category"] == "cirrhosis")]
print(lc.to_string(index=False))

# Save full detail table for downstream use
detail_path = Config.PROCESSED_DIR / "imputation_counts_per_tissue_category.csv"
detail_df.sort_values(["Tissue", "imputed"], ascending=[True, False]).to_csv(detail_path, index=False)
print(f"\nSaved per-(tissue, category) imputation counts: {detail_path}")

=== Top 30 (Tissue, category) cells by NLP-imputed positives ===
                          Tissue           category  original  imputed  total  gain_pct
          Adipose - Subcutaneous           fibrosis       137      133    279      47.7
                  Artery - Aorta             plaque         0      112    183      61.2
              Colon - Transverse    clean_specimens        15       97    112      86.6
         Breast - Mammary Tissue           fibrosis       103       89    219      40.6
                         Stomach    clean_specimens        23       86    112      76.8
                        Pancreas    clean_specimens        22       81    103      78.6
Small Intestine - Terminal Ileum         nodularity         0       76     80      95.0
               Artery - Coronary             plaque         0       67    152      44.1
Small Intestine - Terminal Ileum    clean_specimens        24       63     88      71.6
              Esophagus - Mucosa    clean_specimens    

## 7. Save outputs

Four files written to `data/processed/`:
1. **`meta_data_with_url_imputed.csv`** — full per-sample metadata with a new `Pathology.Categories.Final` column (original GTEx label when present, NLP-imputed otherwise). This is the file `gtex_biomarkers/data.load_raw_data()` consumes so every downstream notebook sees the imputed labels transparently.
2. **`gtex_pathology_matrix_binary.csv`** — wide binary matrix for all 13 tissue groups.
3. **`gtex_pathology_matrix_continuous.csv`** — wide percentage-severity matrix, same shape.
4. **`liver_pathology_labels_imputed.csv`** — liver-only long-form table that keeps the `Pathology.Categories.Final` schema NB04 expects.

Rebuilding any of these is idempotent — re-running this notebook overwrites them.

In [9]:
# Per-sample imputed meta (canonical input for downstream notebooks).
pred_str_all = df_meta_url["pred_cats"].apply(
    lambda s: ", ".join(sorted(s)) if s else np.nan
)
imputed_meta = df_meta_url.copy()
imputed_meta["Pathology.Categories.Final"] = (
    imputed_meta["Pathology.Categories"].fillna(pred_str_all)
)
imputed_meta = imputed_meta.drop(columns=["pred_cats", "pred_sev", "gt_set", "tissue_group"],
                                 errors="ignore")
meta_path = Config.PROCESSED_DIR / "meta_data_with_url_imputed.csv"
imputed_meta.to_csv(meta_path, index=False)
print(f"Saved {meta_path}  ({len(imputed_meta)} rows, "
      f"{imputed_meta['Pathology.Categories.Final'].notna().sum()} with labels)")

binary_path = Config.PROCESSED_DIR / "gtex_pathology_matrix_binary.csv"
continuous_path = Config.PROCESSED_DIR / "gtex_pathology_matrix_continuous.csv"
binary_df.to_csv(binary_path, index=False)
continuous_df.to_csv(continuous_path, index=False)
print(f"Saved {binary_path}")
print(f"Saved {continuous_path}")

# Legacy liver-only long-form (for NB04)
liver = df_meta_url[df_meta_url["Tissue"].astype(str) == "Liver"].copy()
liver["pred_str"] = liver["pred_cats"].apply(lambda s: ", ".join(sorted(s)) if s else np.nan)
liver["Pathology.Categories.Final"] = liver["Pathology.Categories"].fillna(liver["pred_str"])
liver["SUBJID"] = liver["Subject.ID"]
legacy_cols = ["Tissue.Sample.ID", "Tissue", "Subject.ID", "Sex", "Age.Bracket", "Hardy.Scale",
               "Pathology.Categories", "Pathology.Notes", "url", "Pathology.Categories.Final", "SUBJID"]
liver_out = liver[legacy_cols]
liver_path = Config.PROCESSED_DIR / "liver_pathology_labels_imputed.csv"
liver_out.to_csv(liver_path, index=False)
print(f"Saved {liver_path}  ({len(liver_out)} rows)")

Saved /Users/rsinha/Library/CloudStorage/OneDrive-SanfordBurnhamPrebysMedicalDiscoveryInstitute/Desktop/gtex_gene_expression/data/processed/meta_data_with_url_imputed.csv  (25713 rows, 10078 with labels)
Saved /Users/rsinha/Library/CloudStorage/OneDrive-SanfordBurnhamPrebysMedicalDiscoveryInstitute/Desktop/gtex_gene_expression/data/processed/gtex_pathology_matrix_binary.csv
Saved /Users/rsinha/Library/CloudStorage/OneDrive-SanfordBurnhamPrebysMedicalDiscoveryInstitute/Desktop/gtex_gene_expression/data/processed/gtex_pathology_matrix_continuous.csv
Saved /Users/rsinha/Library/CloudStorage/OneDrive-SanfordBurnhamPrebysMedicalDiscoveryInstitute/Desktop/gtex_gene_expression/data/processed/liver_pathology_labels_imputed.csv  (610 rows)
